# Mechanism analysis: first/last anchoring? (read-only, S=7, seed 42)

Notebook 37 showed a striking pattern: models trained on S7_k02 execute a singleton TWICE (A → AA, B → BB,
C → CC = unchanged), and on k=3 they return a two-operation result. Hypothesis: with short training instructions,
the model organises the computation around the FIRST and the LAST real operation. A singleton has first = last
(executed twice), and a longer instruction loses its middle operation(s).

Two read-only checks; nothing is trained:
1. **Which operation is dropped** (from the saved predictions): for held-out longer instructions whose prediction
   equals the word with one operation removed, the dropped position. The hypothesis predicts MIDDLE.
2. **Attention probe** (loads the LAST checkpoints): how much attention the answer positions and the input cells pay
   to the FIRST / MIDDLE / LAST real operation and to the `<identity>` fillers, per layer and per head. This covers
   S7_k02, S7_k0234 and S7_k02345 (the success reference), for AR34, query AR, causal encoder and original AR.

A CPU runtime works but is slow; a GPU makes the probe much faster (set `DEVICE = "cuda"`).
**Send back `Shortcut_Analysis_S7_short_k_Review.ipynb` and `Attention_Probe_Review.ipynb`.**


In [ ]:
# --- 2. Bootstrap: locate or clone the PRIVATE repository (credentials are never printed or stored) ---
import base64, importlib.util, os, pathlib, subprocess, sys

REPO_URL = "https://github.com/zhexuandliu/Decompositionality.git"
REPO_BRANCH = "main"
IN_COLAB = "google.colab" in sys.modules or "COLAB_RELEASE_TAG" in os.environ

def _find_repo():
    candidates = [pathlib.Path("/content/Decompositionality"), pathlib.Path.cwd(), *pathlib.Path.cwd().parents]
    for p in candidates:
        if (p / "src" / "decompositionality").is_dir():
            return p.resolve()
    return None

REPO_DIR = _find_repo()
if REPO_DIR is None:
    token = None
    if IN_COLAB:
        try:
            from google.colab import userdata
            token = userdata.get("GITHUB_TOKEN")
        except Exception:
            token = None
    if not token:
        import getpass
        token = getpass.getpass("GitHub token with read access to the private repo (input hidden): ")
    # Pass the credential through environment-scoped git config (not the URL, not the command line,
    # not persisted in .git/config).
    _hdr = "AUTHORIZATION: basic " + base64.b64encode(f"x-access-token:{token}".encode()).decode()
    _env = dict(os.environ, GIT_TERMINAL_PROMPT="0", GIT_CONFIG_COUNT="1",
                GIT_CONFIG_KEY_0="http.https://github.com/.extraheader", GIT_CONFIG_VALUE_0=_hdr)
    REPO_DIR = pathlib.Path("/content/Decompositionality" if IN_COLAB else "Decompositionality").resolve()
    _r = subprocess.run(["git", "clone", "--depth", "1", "--branch", REPO_BRANCH, REPO_URL, str(REPO_DIR)],
                        env=_env, capture_output=True, text=True)
    del token, _hdr, _env
    if _r.returncode != 0:
        raise RuntimeError("git clone failed (check the GITHUB_TOKEN secret / repo access):\n" + _r.stderr[-2000:])

if str(REPO_DIR / "src") not in sys.path:
    sys.path.insert(0, str(REPO_DIR / "src"))
for _mod in ("numpy", "pandas", "matplotlib", "torch"):
    if importlib.util.find_spec(_mod) is None:  # Colab already ships these; never reinstall torch/CUDA here
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", _mod], check=True)
REPO_COMMIT = subprocess.run(["git", "-C", str(REPO_DIR), "rev-parse", "HEAD"], capture_output=True, text=True).stdout.strip()
print("Repository:", REPO_DIR, "| commit:", REPO_COMMIT or "unknown")


In [ ]:
# --- CONFIGURATION -------------------------------------------------------------------------------------------------
MODE = "full"
SEED = 42
USE_DRIVE = IN_COLAB
RESULTS_ROOT = os.environ.get("DECOMP_RESULTS_ROOT") or (
    "/content/drive/MyDrive/decompositionality_results" if USE_DRIVE else str(REPO_DIR / "results"))
import torch
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
# -----------------------------------------------------------------------------------------------------------------
if USE_DRIVE and IN_COLAB and RESULTS_ROOT.startswith("/content/drive"):
    from google.colab import drive
    drive.mount("/content/drive")
import pandas as pd
pd.set_option("display.max_rows", 500); pd.set_option("display.max_columns", 60); pd.set_option("display.width", 250)
from decompositionality import shortcut_analysis as SA, attn_probe as AP
print("device:", DEVICE)


In [ ]:
# --- 1. Predictions: outcome classes + which operation is dropped (table 5) ---------------------------------------
rp1 = SA.analyse(RESULTS_ROOT, MODE, SEED)
print("REVIEW FILE TO SEND BACK:", rp1)
if IN_COLAB and rp1:
    from google.colab import files
    files.download(str(rp1))


In [ ]:
# --- 2. Attention probe (loads the LAST checkpoints) ---------------------------------------------------------------
rp2 = AP.probe(RESULTS_ROOT, MODE, SEED, device=DEVICE)
print("REVIEW FILE TO SEND BACK:", rp2)
if IN_COLAB and rp2:
    from google.colab import files
    files.download(str(rp2))
